In [21]:
import os
import json
import joblib

import pandas as pd

from xgboost import XGBClassifier

In [22]:
X_final = df_new.drop(columns=["Exited"])
y_final = df_new["Exited"]

if "AgeGroup" in X_final.columns:
    X_final = X_final.drop(columns=["AgeGroup"])

X_final = pd.get_dummies(
    X_final,
    drop_first=True
)

In [23]:
scale_pos_weight = (
    (y_final == 0).sum()
    /
    (y_final == 1).sum()
)

best_xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,

    learning_rate=0.2,
    max_depth=4,
    min_child_weight=7,
    gamma=0.1,
    subsample=1.0,
    colsample_bytree=0.9,
    n_estimators=100,

    scale_pos_weight=scale_pos_weight
)

best_xgb.fit(X_final, y_final)

print("Final XGBoost trained successfully.")

Final XGBoost trained successfully.


In [24]:
os.makedirs("artifacts", exist_ok=True)

joblib.dump(
    best_xgb,
    "artifacts/final_model.pkl"
)

with open("artifacts/model_columns.json", "w") as f:
    json.dump(
        list(X_final.columns),
        f,
        indent=4
    )

model_config = {
    "ML Model Used": "Tuned XGBoost",
    "Threshold": 0.55,
    "Target": "Exited",
    "Precision": 0.5424,
    "Recall": 0.7076,
    "F1 Score": 0.6141,
    "ROC AUC": 0.8626,
    "Purpose": "Customer Churn Prediction",
    "Class Imbalance Handling": "scale_pos_weight",
    "SHAP Compatible": True
}

with open(
    "artifacts/model_config.json",
    "w"
) as f:
    json.dump(
        model_config,
        f,
        indent=4
    )

print("Artifacts saved successfully.")
print(f"Features : {len(X_final.columns)}")

Artifacts saved successfully.
Features : 11


In [25]:
loaded_model = joblib.load(
    "artifacts/final_model.pkl"
)

with open("artifacts/model_columns.json") as f:
    loaded_columns = json.load(f)

with open("artifacts/model_config.json") as f:
    loaded_config = json.load(f)

print(type(loaded_model))
print()

print(loaded_config)
print()

print(f"Number of features: {len(loaded_columns)}")

<class 'xgboost.sklearn.XGBClassifier'>

{'ML Model Used': 'Tuned XGBoost', 'Threshold': 0.55, 'Target': 'Exited', 'Precision': 0.5424, 'Recall': 0.7076, 'F1 Score': 0.6141, 'ROC AUC': 0.8626, 'Purpose': 'Customer Churn Prediction', 'Class Imbalance Handling': 'scale_pos_weight', 'SHAP Compatible': True}

Number of features: 11
